In [1]:
import jiwer
import pandas as pd
import numpy as np
from pathlib import Path
from google import genai
from google.genai import types
from bs4 import BeautifulSoup
import time
import json
import re

In [2]:
with open("apikey.txt", "r") as f:
    api_key = f.read().strip()

client = genai.Client(api_key=api_key)

# Class ID to label mapping (derived from cross-referencing YOLO outputs with ALTO XML)
YOLO_CLASS_MAP = {
    3: "GraphicZone",
    4: "GraphicZone-Decoration",
    6: "GraphicZone-Head",
    12: "MainZone-Head",
    16: "MainZone-P",
    18: "MainZone-Sp",
    22: "NumberingZone",
}

# All pages share these dimensions
PAGE_WIDTH = 1559
PAGE_HEIGHT = 2088

In [3]:
# --- Parsing functions ---

def parse_alto_xml(xml_path):
    """Extract ground-truth bounding boxes and labels from ALTO XML."""
    with open(xml_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "xml")

    # Build REGION_TYPE_ID -> label mapping from OtherTag elements
    tag_map = {}
    for tag in soup.find_all("OtherTag"):
        if tag.get("TYPE") == "region":
            tag_map[tag["ID"]] = tag["LABEL"]

    boxes = []
    for block in soup.find_all("TextBlock"):
        tag_ref = block.get("TAGREFS", "")
        label = tag_map.get(tag_ref, "Unknown")
        hpos = float(block["HPOS"])
        vpos = float(block["VPOS"])
        width = float(block["WIDTH"])
        height = float(block["HEIGHT"])
        boxes.append({
            "label": label,
            "x_min": hpos,
            "y_min": vpos,
            "x_max": hpos + width,
            "y_max": vpos + height,
        })
    return boxes


def parse_yolo_txt(txt_path, img_w=PAGE_WIDTH, img_h=PAGE_HEIGHT):
    """Convert YOLO normalized format to pixel bounding boxes."""
    boxes = []
    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            class_id = int(parts[0])
            cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x_min = (cx - w / 2) * img_w
            y_min = (cy - h / 2) * img_h
            x_max = (cx + w / 2) * img_w
            y_max = (cy + h / 2) * img_h
            label = YOLO_CLASS_MAP.get(class_id, f"class_{class_id}")
            boxes.append({
                "label": label,
                "x_min": x_min,
                "y_min": y_min,
                "x_max": x_max,
                "y_max": y_max,
            })
    return boxes


print("Parsing functions defined.")

Parsing functions defined.


In [4]:
# --- Detection metrics ---

def compute_iou(box_a, box_b):
    """Compute IoU between two boxes (each a dict with x_min, y_min, x_max, y_max)."""
    x1 = max(box_a["x_min"], box_b["x_min"])
    y1 = max(box_a["y_min"], box_b["y_min"])
    x2 = min(box_a["x_max"], box_b["x_max"])
    y2 = min(box_a["y_max"], box_b["y_max"])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a["x_max"] - box_a["x_min"]) * (box_a["y_max"] - box_a["y_min"])
    area_b = (box_b["x_max"] - box_b["x_min"]) * (box_b["y_max"] - box_b["y_min"])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def match_boxes(gt_boxes, pred_boxes, iou_threshold=0.5):
    """Greedy matching of predicted boxes to ground-truth boxes.
    Returns matched pairs, unmatched GT (missed), and unmatched pred (false positives)."""
    iou_matrix = np.zeros((len(gt_boxes), len(pred_boxes)))
    for i, gt in enumerate(gt_boxes):
        for j, pred in enumerate(pred_boxes):
            iou_matrix[i, j] = compute_iou(gt, pred)

    matched = []
    used_gt = set()
    used_pred = set()

    # Greedy: pick highest IoU pairs first
    while True:
        if iou_matrix.size == 0:
            break
        max_idx = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)
        max_iou = iou_matrix[max_idx]
        if max_iou < iou_threshold:
            break
        i, j = max_idx
        matched.append({"gt": gt_boxes[i], "pred": pred_boxes[j], "iou": max_iou})
        used_gt.add(i)
        used_pred.add(j)
        iou_matrix[i, :] = 0
        iou_matrix[:, j] = 0

    missed = [gt_boxes[i] for i in range(len(gt_boxes)) if i not in used_gt]
    false_pos = [pred_boxes[j] for j in range(len(pred_boxes)) if j not in used_pred]
    return matched, missed, false_pos


def compute_page_metrics(gt_boxes, pred_boxes, iou_threshold=0.5):
    """Compute precision, recall, F1, and mean IoU for a single page."""
    matched, missed, false_pos = match_boxes(gt_boxes, pred_boxes, iou_threshold)
    tp = len(matched)
    fp = len(false_pos)
    fn = len(missed)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    mean_iou = np.mean([m["iou"] for m in matched]) if matched else 0.0
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "mean_iou": round(mean_iou, 4),
        "true_pos": tp,
        "false_pos": fp,
        "missed": fn,
    }


print("Detection metrics functions defined.")

Detection metrics functions defined.


In [5]:
# --- LLM-as-judge ---

def format_boxes_for_llm(boxes):
    """Format bounding boxes as a readable string for the LLM prompt."""
    lines = []
    for i, b in enumerate(boxes):
        lines.append(f"  {i+1}. [{b['label']}] x:({b['x_min']:.0f}-{b['x_max']:.0f}) y:({b['y_min']:.0f}-{b['y_max']:.0f})")
    return "\n".join(lines) if lines else "  (none)"


def llm_judge_layout(gt_boxes, pred_boxes, page_name, max_retries=3):
    """Ask the LLM to evaluate layout detection quality. Returns parsed JSON or error."""
    gt_str = format_boxes_for_llm(gt_boxes)
    pred_str = format_boxes_for_llm(pred_boxes)

    system_prompt = (
        "You are an expert document layout analysis evaluator. "
        "You will compare predicted region bounding boxes against ground-truth regions. "
        "Respond with ONLY valid JSON matching this schema:\n"
        '{"layout_score": <int 1-10>, "missed_regions": <int>, "extra_regions": <int>, '
        '"mislocalized_regions": <int>, "label_mismatches": <int>, "rationale": "<brief text>"}\n'
        "Scoring guide: 10 = perfect match, 7-9 = minor issues, 4-6 = significant errors, 1-3 = severe failures."
    )

    user_prompt = (
        f"Page: {page_name} (dimensions: {PAGE_WIDTH}x{PAGE_HEIGHT} pixels)\n\n"
        f"Ground-truth regions ({len(gt_boxes)} total):\n{gt_str}\n\n"
        f"Predicted regions ({len(pred_boxes)} total):\n{pred_str}\n\n"
        "Evaluate how well the predictions match the ground truth. Consider: "
        "region count accuracy, spatial overlap, label correctness, and missed/extra detections."
    )

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    temperature=0.2,
                    max_output_tokens=400,
                ),
            )
            text = response.text.strip()
            # Strip markdown code fences if present
            text = re.sub(r"^```(?:json)?\s*", "", text)
            text = re.sub(r"\s*```$", "", text)
            return json.loads(text)
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                wait = 2 ** attempt * 15  # 15s, 30s, 60s
                print(f"  Rate limited on {page_name}, waiting {wait}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                return {"layout_score": None, "error": str(e)}
    return {"layout_score": None, "error": "Max retries exceeded"}


print("LLM judge function defined.")

LLM judge function defined.


In [6]:
# --- Run the evaluation pipeline ---

def run_evaluation_pipeline(ref_dir, pred_dir, iou_threshold=0.5):
    """Run both traditional detection metrics and LLM-as-judge on all pages."""
    results = []
    ref_files = sorted(Path(ref_dir).glob("*.xml"))

    for ref_path in ref_files:
        page_name = ref_path.stem
        pred_path = Path(pred_dir) / f"{page_name}.txt"

        if not pred_path.exists():
            print(f"Skipping {page_name}: no prediction file found")
            continue

        print(f"Evaluating {page_name}...")
        gt_boxes = parse_alto_xml(ref_path)
        pred_boxes = parse_yolo_txt(pred_path)

        # Traditional metrics
        metrics = compute_page_metrics(gt_boxes, pred_boxes, iou_threshold)

        # LLM judge
        llm_result = llm_judge_layout(gt_boxes, pred_boxes, page_name)
        time.sleep(13)  # Stay under 5 requests/minute free tier

        results.append({
            "page": page_name,
            "gt_count": len(gt_boxes),
            "pred_count": len(pred_boxes),
            **metrics,
            "llm_score": llm_result.get("layout_score"),
            "llm_missed": llm_result.get("missed_regions"),
            "llm_extra": llm_result.get("extra_regions"),
            "llm_mislocalized": llm_result.get("mislocalized_regions"),
            "llm_label_mismatches": llm_result.get("label_mismatches"),
            "llm_rationale": llm_result.get("rationale", llm_result.get("error", "")),
        })

    df = pd.DataFrame(results)
    df.to_csv("evaluation_results.csv", index=False)
    return df


df_results = run_evaluation_pipeline("TEI_directory6", "pipeline_dist_output")

Evaluating page_0020...
Evaluating page_0021...
Evaluating page_0022...
Evaluating page_0023...
Evaluating page_0024...
Evaluating page_0025...
Evaluating page_0026...
Evaluating page_0027...
Evaluating page_0028...
Evaluating page_0029...
Evaluating page_0030...


In [7]:
# --- Display results ---

print("=== Traditional Detection Metrics ===")
display(df_results[["page", "gt_count", "pred_count", "precision", "recall", "f1", "mean_iou", "true_pos", "false_pos", "missed"]])

print("\n=== LLM Judge Scores ===")
display(df_results[["page", "llm_score", "llm_missed", "llm_extra", "llm_mislocalized", "llm_label_mismatches", "llm_rationale"]])

=== Traditional Detection Metrics ===


,page,gt_count,pred_count,precision,recall,f1,mean_iou,true_pos,false_pos,missed
0,page_0020,15,15,1.0,1.0,1.0,0.9943,15,0,0
1,page_0021,11,11,1.0,1.0,1.0,0.9947,11,0,0
2,page_0022,13,13,1.0,1.0,1.0,0.9941,13,0,0
3,page_0023,12,12,1.0,1.0,1.0,0.9935,12,0,0
4,page_0024,16,16,1.0,1.0,1.0,0.9928,16,0,0
5,page_0025,11,11,1.0,1.0,1.0,0.9914,11,0,0
6,page_0026,15,15,1.0,1.0,1.0,0.9943,15,0,0
7,page_0027,14,14,1.0,1.0,1.0,0.9831,14,0,0
8,page_0028,16,16,1.0,1.0,1.0,0.9948,16,0,0
9,page_0029,15,15,1.0,1.0,1.0,0.9894,15,0,0



=== LLM Judge Scores ===


,page,llm_score,llm_missed,llm_extra,llm_mislocalized,llm_label_mismatches,llm_rationale
0,page_0020,None,None,None,None,None,Expecting property name enclosed in double quo...
1,page_0021,None,None,None,None,None,"Expecting ',' delimiter: line 2 column 20 (cha..."
2,page_0022,None,None,None,None,None,"Expecting ',' delimiter: line 2 column 20 (cha..."
3,page_0023,None,None,None,None,None,Expecting property name enclosed in double quo...
4,page_0024,None,None,None,None,None,Expecting property name enclosed in double quo...
5,page_0025,None,None,None,None,None,"Expecting ',' delimiter: line 2 column 20 (cha..."
6,page_0026,None,None,None,None,None,"Expecting ',' delimiter: line 2 column 21 (cha..."
7,page_0027,None,None,None,None,None,"Expecting ',' delimiter: line 2 column 21 (cha..."
8,page_0028,None,None,None,None,None,"Expecting ',' delimiter: line 2 column 20 (cha..."
9,page_0029,None,None,None,None,None,Expecting property name enclosed in double quo...


In [9]:
# --- Correlation analysis ---
import matplotlib.pyplot as plt

# Filter to pages where LLM returned a valid score
df_valid = df_results.dropna(subset=["llm_score"]).copy()

if len(df_valid) >= 2:
    corr_f1 = df_valid["f1"].corr(df_valid["llm_score"])
    corr_iou = df_valid["mean_iou"].corr(df_valid["llm_score"])
    print(f"Pearson correlation — F1 vs LLM score: {corr_f1:.3f}")
    print(f"Pearson correlation — Mean IoU vs LLM score: {corr_iou:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(df_valid["f1"], df_valid["llm_score"], s=80, edgecolors="black")
    for _, row in df_valid.iterrows():
        axes[0].annotate(row["page"], (row["f1"], row["llm_score"]),
                         fontsize=8, ha="left", va="bottom")
    axes[0].set_xlabel("F1 Score (IoU@0.5)")
    axes[0].set_ylabel("LLM Judge Score (1-10)")
    axes[0].set_title(f"F1 vs LLM Score (r={corr_f1:.3f})")
    axes[0].set_xlim(-0.05, 1.05)
    axes[0].set_ylim(0, 11)

    axes[1].scatter(df_valid["mean_iou"], df_valid["llm_score"], s=80, edgecolors="black", color="orange")
    for _, row in df_valid.iterrows():
        axes[1].annotate(row["page"], (row["mean_iou"], row["llm_score"]),
                         fontsize=8, ha="left", va="bottom")
    axes[1].set_xlabel("Mean IoU (matched pairs)")
    axes[1].set_ylabel("LLM Judge Score (1-10)")
    axes[1].set_title(f"Mean IoU vs LLM Score (r={corr_iou:.3f})")
    axes[1].set_xlim(-0.05, 1.05)
    axes[1].set_ylim(0, 11)

    plt.tight_layout()
    plt.savefig("correlation_plots.png", dpi=150)
    plt.show()
else:
    print("Not enough valid LLM scores to compute correlation.")

Not enough valid LLM scores to compute correlation.


In [ ]:
# --- Summary statistics ---

print("=== Aggregate Metrics ===")
print(f"Mean F1:       {df_results['f1'].mean():.3f}")
print(f"Mean Precision:{df_results['precision'].mean():.3f}")
print(f"Mean Recall:   {df_results['recall'].mean():.3f}")
print(f"Mean IoU:      {df_results['mean_iou'].mean():.3f}")

if df_valid is not None and len(df_valid) > 0:
    print(f"Mean LLM Score:{df_valid['llm_score'].mean():.1f} / 10")

print("\n=== Per-page Summary ===")
display(df_results[["page", "f1", "mean_iou", "llm_score"]].to_string(index=False))